# Extension 3: Sequential Unlearning — GA vs PO

**Goal:** Measure metric degradation as 20 forget-set IDs are unlearned sequentially in 3 rounds using 2 contrasting methods (GA, PO).

**Design:**
- Round 1: Unlearn batch 1 (7 IDs) from stage1
- Round 2: Unlearn batch 2 (7 IDs) from round1 checkpoint
- Round 3: Unlearn batch 3 (6 IDs) from round2 checkpoint

**Output:** Metrics table showing rouge_l, exact_match, mia_mink, ks_test_pval, avg_mu across rounds 

## Section 0: Setup

In [35]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
import subprocess
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'

# Skip clone if repo already exists and is valid
if not Path(PROJECT_ROOT).exists() or not (Path(PROJECT_ROOT) / 'FIUBench').exists():
    print("Cloning FIUBench_Reproducing...\n")
    
    # Use public clone (or set GH_TOKEN env var if private)
    clone_url = "https://github.com/akashyall34/FIUBench_Reproducing.git"
    
    result = subprocess.run(
        f"git clone {clone_url} {PROJECT_ROOT}",
        shell=True, capture_output=True, text=True
    )
    
    if result.returncode == 0:
        print("✅ Repository cloned successfully")
    else:
        print("❌ Clone failed:")
        print(result.stderr)
        raise RuntimeError(f"Failed to clone repo")
else:
    print(f"✅ Repository already exists at {PROJECT_ROOT}")

# Verify structure
assert (Path(PROJECT_ROOT) / 'FIUBench').exists(), "❌ FIUBench subdirectory not found"
print(f"✅ FIUBench directory confirmed")

✅ Repository already exists at /content/FIUBench_Reproducing
✅ FIUBench directory confirmed


In [37]:
import subprocess
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'

print(f"Pulling latest changes from GitHub...\n")

# Stash local changes first to avoid conflicts
print("Stashing local changes...")
result = subprocess.run(
    "git stash",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)
if result.returncode == 0 and result.stdout.strip():
    print(result.stdout)
else:
    print("(no local changes to stash)")

# Now pull
result = subprocess.run(
    "git pull",
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    shell=True
)

if result.returncode == 0:
    print("✅ Repository updated")
    print(result.stdout)
else:
    print("❌ Pull failed")
    print(result.stderr)

Pulling latest changes from GitHub...

Stashing local changes...
No local changes to save

✅ Repository updated
Already up to date.



In [22]:
import subprocess, sys

deps = [
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1",
    "transformers==4.48.0",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "python-dotenv",
    "openai",
    "hydra-core",
    "scipy",
    "deepspeed",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

import transformers
import torch
print(f"✅ torch={torch.__version__}  transformers={transformers.__version__}")

✅ torch=2.4.1+cu121  transformers=4.48.0


## Copy Stage 1

In [38]:
import os, sys, re
from pathlib import Path

# ── Setup paths ───────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_ROOT = '/content/FIUBench_Reproducing' if IN_COLAB else '/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/FIUBench_Reproducing'
STAGE1_LOCAL = '/content/stage1_final' if IN_COLAB else f'{PROJECT_ROOT}/stage1_final'
STAGE1_DRIVE = '/content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints'
FIUBENCH_DIR = f'{PROJECT_ROOT}/FIUBench'

os.environ['WANDB_MODE'] = 'disabled'
os.environ['HYDRA_FULL_ERROR'] = '1'

# ── 1. Copy Stage 1 from Drive ─────────────────────────────────────────────────
if IN_COLAB:
    if not Path(STAGE1_LOCAL).exists() or not list(Path(STAGE1_LOCAL).glob('*.safetensors')):
        print("Copying Stage 1 from Drive...")
        Path(STAGE1_LOCAL).mkdir(parents=True, exist_ok=True)
        ret = os.system(f'rsync -ah --progress {STAGE1_DRIVE}/ {STAGE1_LOCAL}/')
        assert ret == 0, "rsync failed"

safetensors = list(Path(STAGE1_LOCAL).glob('*.safetensors'))
assert safetensors, f"No safetensors in {STAGE1_LOCAL}"
print(f"✅ Stage 1 ready: {[f.name for f in safetensors]}")

# ── 2. Patch forget.py ─────────────────────────────────────────────────────────
fg_py = Path(FIUBENCH_DIR) / 'forget.py'
assert fg_py.exists(), f"❌ forget.py not found at {fg_py}"

src = fg_py.read_text()
patched = src
patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')
patched = patched.replace('.to(torch.float16)', '')
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)
if patched != src:
    fg_py.write_text(patched)
    print("✅ Patched forget.py")
else:
    print("✅ forget.py already patched")

assert 'torch_dtype=torch.bfloat16' in fg_py.read_text(), "bfloat16 patch missing"
assert 'mixed_precision="bf16"' in fg_py.read_text(), "mixed_precision patch missing"

# ── 3. Patch modeling_llava.py ─────────────────────────────────────────────────
import glob
llava_candidates = glob.glob(
    '/usr/local/lib/python*/dist-packages/transformers/models/llava/modeling_llava.py'
)
if llava_candidates:
    llava_path = llava_candidates[0]
    llava_src = Path(llava_path).read_text()
    llava_patched = re.sub(
        r'n_image_tokens != n_image_features',
        'False',
        llava_src
    )
    if llava_patched != llava_src:
        Path(llava_path).write_text(llava_patched)
        print(f"✅ Patched modeling_llava.py")
    else:
        print("✅ modeling_llava.py already patched")
else:
    print("⚠️  modeling_llava.py not found — may not be needed for this transformers version")

print(f"\n✅ Setup complete:")
print(f"   PROJECT_ROOT: {PROJECT_ROOT}")
print(f"   FIUBENCH_DIR: {FIUBENCH_DIR}")
print(f"   STAGE1_LOCAL: {STAGE1_LOCAL}")
print(f"\n✅ All patches applied. Ready for Stage 2.")

✅ Stage 1 ready: ['model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors']
✅ forget.py already patched
✅ modeling_llava.py already patched

✅ Setup complete:
   PROJECT_ROOT: /content/FIUBench_Reproducing
   FIUBENCH_DIR: /content/FIUBench_Reproducing/FIUBench
   STAGE1_LOCAL: /content/stage1_final

✅ All patches applied. Ready for Stage 2.


## Section 1: Load & Partition Forget Set

In [39]:
import json
from pathlib import Path
import numpy as np

with open(f'{FIUBENCH_DIR}/dataset/full.json') as f:
    full_data = [json.loads(line) for line in f if line.strip()]
with open(f'{FIUBENCH_DIR}/dataset/split.json') as f:
    splits = json.load(f)

forget5_ids = list(set(splits['forget5']))
forget_data = [d for d in full_data if d['unique_id'] in forget5_ids]

print(f'Total forget5 identities: {len(forget_data)}')

np.random.seed(42)
shuffled = forget_data.copy()
np.random.shuffle(shuffled)

batch1 = shuffled[0:7]
batch2 = shuffled[7:14]
batch3 = shuffled[14:20]

# Cumulative unique_ids per round
CUM_IDS = {
    1: [d['unique_id'] for d in batch1],
    2: [d['unique_id'] for d in batch1 + batch2],
    3: [d['unique_id'] for d in batch1 + batch2 + batch3],
}

print(f'Round 1: {len(CUM_IDS[1])} IDs')
print(f'Round 2: {len(CUM_IDS[2])} IDs')
print(f'Round 3: {len(CUM_IDS[3])} IDs')


Total forget5 identities: 20
Round 1: 7 IDs
Round 2: 14 IDs
Round 3: 20 IDs


## Section 2: Restore Stage1 & Apply Patches

In [40]:
import os
from pathlib import Path

STAGE1_LOCAL = '/content/stage1_final' if IN_COLAB else f'{PROJECT_ROOT}/stage1_final'
STAGE1_DRIVE = '/content/drive/MyDrive/fiubench_checkpoints/stage1_checkpoints'

# Verify stage1 is ready (Cell-7 already copied it from Drive)
stage1_files = list(Path(STAGE1_LOCAL).glob('*.safetensors'))
assert stage1_files, f'No safetensors in {STAGE1_LOCAL}'
print(f'✅ Stage1 ready: {len(stage1_files)} files')

# Verify forget.py patches are in place (Cell-7 already applied them)
fg_py = Path(FIUBENCH_DIR) / 'forget.py'
assert 'torch_dtype=torch.bfloat16' in fg_py.read_text(), "❌ bfloat16 patch missing"
assert 'mixed_precision="bf16"' in fg_py.read_text(), "❌ mixed_precision patch missing"
print('✅ forget.py patches verified')
print('\n✅ All prerequisites ready for sequential unlearning')

✅ Stage1 ready: 2 files
✅ forget.py patches verified

✅ All prerequisites ready for sequential unlearning


## Dataset

In [41]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

PROJECT_ROOT = '/content/FIUBench_Reproducing'

try:
    sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
    sfhq_dir.mkdir(parents=True, exist_ok=True)

    existing = list(sfhq_dir.glob("*.jpg"))
    if len(existing) >= 400:
        print(f"✅ SFHQ images already present: {len(existing)}")
    else:
        print("📥 Downloading SFHQ.zip from Hugging Face...")
        zip_path = hf_hub_download(
            repo_id="gray311/FIUBench",
            filename="SFHQ.zip",
            repo_type="dataset",
            token=os.environ.get("HF_TOKEN"),
        )

        extract_dir = sfhq_dir.parent / "sfhq_extracted"
        extract_dir.mkdir(parents=True, exist_ok=True)

        print(f"📦 Extracting SFHQ.zip...")
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

        found = list(extract_dir.rglob("*.jpg"))
        print(f"Found {len(found)} jpg files")

        copied = 0
        for src in found:
            dst = sfhq_dir / src.name
            if not dst.exists():
                shutil.copy2(src, dst)
                copied += 1

        total = len(list(sfhq_dir.glob("*.jpg")))
        print(f"✅ SFHQ ready: {total} images")

except Exception as e:
    print(f"❌ SFHQ download failed: {e}")
    raise

# Create symlink in FIUBENCH_DIR
sfhq_src = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dst = Path(FIUBENCH_DIR) / "dataset" / "SFHQ"

if sfhq_dst.is_symlink():
    sfhq_dst.unlink()
elif sfhq_dst.exists():
    shutil.rmtree(sfhq_dst)

sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
sfhq_dst.symlink_to(sfhq_src)

n = len(list(sfhq_dst.glob("*.jpg")))
print(f"✅ Symlinked: {n} images")


✅ SFHQ images already present: 1000
✅ Symlinked: 1000 images


In [42]:
from pathlib import Path
import shutil

sfhq_src = Path('/content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ')
sfhq_dst = Path('/content/FIUBench_Reproducing/FIUBench/dataset/SFHQ')

# Verify source exists
if not sfhq_src.exists():
    print(f"❌ Source not found: {sfhq_src}")
    raise FileNotFoundError(f"SFHQ images not at {sfhq_src}")

# Clean up old symlink/directory
if sfhq_dst.is_symlink():
    sfhq_dst.unlink()
elif sfhq_dst.exists():
    shutil.rmtree(sfhq_dst)

# Create symlink
sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
sfhq_dst.symlink_to(sfhq_src)

n = len(list(sfhq_dst.glob("*.jpg")))
if n < 400:
    print(f"⚠️  Only {n} images (expected 400+)")
else:
    print(f"✅ Symlinked: {n} images")

✅ Symlinked: 1000 images


In [43]:
from pathlib import Path
import json

fiubench = Path('/content/FIUBench_Reproducing/FIUBench')
with open(fiubench / 'dataset/full.json') as f:
    data = [json.loads(line) for line in f if line.strip()]
for item in data[:5]:
    p = fiubench / item['image_path']
    print(f"{'✅' if p.exists() else '❌'} {item['image_path']}")


✅ ./dataset/SFHQ/SFHQ_pt1_00044363.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053161.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00055331.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00022936.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053085.jpg


## GA

In [44]:
import subprocess, time, json, os
from pathlib import Path

METHOD = 'ga'
METHOD_LABEL = 'ga'
LR = '2e-5'
PORT = '29500'

results_ga = {}

SPLIT_JSON = Path(FIUBENCH_DIR) / 'dataset' / 'split.json'

def inject_split(round_num, cum_ids):
    with open(SPLIT_JSON) as f:
        s = json.load(f)
    s[f'ext3_r{round_num}'] = cum_ids
    with open(SPLIT_JSON, 'w') as f:
        json.dump(s, f)
    print(f'  Injected ext3_r{round_num} ({len(cum_ids)} IDs) into split.json')

def remove_ext3_splits():
    with open(SPLIT_JSON) as f:
        s = json.load(f)
    for k in [k for k in s if k.startswith('ext3_r')]:
        del s[k]
    with open(SPLIT_JSON, 'w') as f:
        json.dump(s, f)

def run_round(round_num, model_path, save_dir):
    inject_split(round_num, CUM_IDS[round_num])
    cmd = (
        f"cd {FIUBENCH_DIR} && "
        f"torchrun --nproc_per_node=1 --master_port={PORT} forget.py "
        f"--config-name forget_lora "
        f"model_path={model_path} "
        f"save_dir={save_dir} "
        f"split=ext3_r{round_num} "
        f"forget_loss={METHOD_LABEL} "
        f"lr={LR} "
        f"batch_size=8 "
        f"num_epochs=8 "
        f"seed=233 "
        f"overwrite_dir=true"
    )
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode, int(time.time() - t0)

# ── Round 1 ───────────────────────────────────────────────────────────────────
print('\n' + '='*70)
print(f'[GA] ROUND 1: {len(CUM_IDS[1])} IDs from stage1')
print('='*70)

r1_dir = Path('/content/ext3_ga_r1') if IN_COLAB else Path('/tmp/ext3_ga_r1')
r1_dir.mkdir(parents=True, exist_ok=True)

ret, elapsed = run_round(1, STAGE1_LOCAL, r1_dir)
results_ga[1] = {'cumulative_ids': len(CUM_IDS[1]), 'model_path': str(r1_dir)}
print(f'\n{"✅" if ret==0 else "❌"} Round 1 ({elapsed}s)')

# ── Round 2 ───────────────────────────────────────────────────────────────────
if ret == 0:
    print('\n' + '='*70)
    print(f'[GA] ROUND 2: {len(CUM_IDS[2])} IDs from round1 checkpoint')
    print('='*70)

    r2_dir = Path('/content/ext3_ga_r2') if IN_COLAB else Path('/tmp/ext3_ga_r2')
    r2_dir.mkdir(parents=True, exist_ok=True)

    ret, elapsed = run_round(2, r1_dir, r2_dir)
    results_ga[2] = {'cumulative_ids': len(CUM_IDS[2]), 'model_path': str(r2_dir)}
    print(f'\n{"✅" if ret==0 else "❌"} Round 2 ({elapsed}s)')

    # ── Round 3 ───────────────────────────────────────────────────────────────
    if ret == 0:
        print('\n' + '='*70)
        print(f'[GA] ROUND 3: {len(CUM_IDS[3])} IDs from round2 checkpoint')
        print('='*70)

        r3_dir = Path('/content/ext3_ga_r3') if IN_COLAB else Path('/tmp/ext3_ga_r3')
        r3_dir.mkdir(parents=True, exist_ok=True)

        ret, elapsed = run_round(3, r2_dir, r3_dir)
        results_ga[3] = {'cumulative_ids': len(CUM_IDS[3]), 'model_path': str(r3_dir)}
        print(f'\n{"✅" if ret==0 else "❌"} Round 3 ({elapsed}s)')

remove_ext3_splits()
print('\n✅ GA sequential unlearning complete')



[GA] ROUND 1: 7 IDs from stage1
  Injected ext3_r1 (7 IDs) into split.json
2026-04-27 22:23:20.513205: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 22:23:20.531660: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777328600.553217   26096 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777328600.559956   26096 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777328600.576383   26096 computation_placer.cc:177] co

## Section 4: Sequential Unlearning — PO

In [45]:
import subprocess, time, json, os
from pathlib import Path

METHOD = 'po'
METHOD_LABEL = 'idk'
LR = '3e-4'
PORT = '29503'


results_po = {}

SPLIT_JSON = Path(FIUBENCH_DIR) / 'dataset' / 'split.json'

def inject_split(round_num, cum_ids):
    with open(SPLIT_JSON) as f:
        s = json.load(f)
    s[f'ext3_r{round_num}'] = cum_ids
    with open(SPLIT_JSON, 'w') as f:
        json.dump(s, f)
    print(f'  Injected ext3_r{round_num} ({len(cum_ids)} IDs) into split.json')

def remove_ext3_splits():
    with open(SPLIT_JSON) as f:
        s = json.load(f)
    for k in [k for k in s if k.startswith('ext3_r')]:
        del s[k]
    with open(SPLIT_JSON, 'w') as f:
        json.dump(s, f)

def run_round(round_num, model_path, save_dir):
    inject_split(round_num, CUM_IDS[round_num])
    cmd = (
        f"cd {FIUBENCH_DIR} && "
        f"torchrun --nproc_per_node=1 --master_port={PORT} forget.py "
        f"--config-name forget_lora "
        f"model_path={model_path} "
        f"save_dir={save_dir} "
        f"split=ext3_r{round_num} "
        f"forget_loss={METHOD_LABEL} "
        f"lr={LR} "
        f"batch_size=8 "
        f"num_epochs=8 "
        f"seed=233 "
        f"overwrite_dir=true"
    )
    t0 = time.time()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    return proc.returncode, int(time.time() - t0)

# ── Round 1 ───────────────────────────────────────────────────────────────────
print('\n' + '='*70)
print(f'[PO] ROUND 1: {len(CUM_IDS[1])} IDs from stage1')
print('='*70)

r1_dir = Path('/content/ext3_po_r1') if IN_COLAB else Path('/tmp/ext3_po_r1')
r1_dir.mkdir(parents=True, exist_ok=True)

ret, elapsed = run_round(1, STAGE1_LOCAL, r1_dir)
results_po[1] = {'cumulative_ids': len(CUM_IDS[1]), 'model_path': str(r1_dir)}
print(f'\n{"✅" if ret==0 else "❌"} Round 1 ({elapsed}s)')

# ── Round 2 ───────────────────────────────────────────────────────────────────
if ret == 0:
    print('\n' + '='*70)
    print(f'[PO] ROUND 2: {len(CUM_IDS[2])} IDs from round1 checkpoint')
    print('='*70)

    r2_dir = Path('/content/ext3_po_r2') if IN_COLAB else Path('/tmp/ext3_po_r2')
    r2_dir.mkdir(parents=True, exist_ok=True)

    ret, elapsed = run_round(2, r1_dir, r2_dir)
    results_po[2] = {'cumulative_ids': len(CUM_IDS[2]), 'model_path': str(r2_dir)}
    print(f'\n{"✅" if ret==0 else "❌"} Round 2 ({elapsed}s)')

    # ── Round 3 ───────────────────────────────────────────────────────────────
    if ret == 0:
        print('\n' + '='*70)
        print(f'[PO] ROUND 3: {len(CUM_IDS[3])} IDs from round2 checkpoint')
        print('='*70)

        r3_dir = Path('/content/ext3_po_r3') if IN_COLAB else Path('/tmp/ext3_po_r3')
        r3_dir.mkdir(parents=True, exist_ok=True)

        ret, elapsed = run_round(3, r2_dir, r3_dir)
        results_po[3] = {'cumulative_ids': len(CUM_IDS[3]), 'model_path': str(r3_dir)}
        print(f'\n{"✅" if ret==0 else "❌"} Round 3 ({elapsed}s)')

remove_ext3_splits()
print('\n✅ PO sequential unlearning complete')



[PO] ROUND 1: 7 IDs from stage1
  Injected ext3_r1 (7 IDs) into split.json
2026-04-27 22:30:26.720484: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 22:30:26.739126: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777329026.760375   30919 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777329026.767300   30919 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777329026.784594   30919 computation_placer.cc:177] co

## Section 5: Evaluate Each Checkpoint

Run eval_accurate_*.py for each round checkpoint and record metrics.

In [46]:
============================================================
RESULTS: /content/ext3_ga_r3
============================================================
  KS-Test p-val ↑              0.40%
  Exact Match ↓                0.00%
  MINK ↓                       0.00%
  APE ↓                        1.25%
  Avg FQ ↑                     50.01%
  ROUGE-L (retain) ↑           50.76%
  Truth Ratio (retain) ↑       8.67%
  ACC (retain) ↑               21.71%
  Avg MU ↑                     16.56%


[Baseline (Stage1)]
2026-04-27 22:38:08.636417: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-27 22:38:08.653803: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777329488.675455   35865 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777329488.682184   35865 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777329488.698859   35865 computation_placer.cc:177] computation placer already registered. Please check linka